# Kaggle Complete Notebook: EagleVision Phase 1 Geometric Depth Improvement

This notebook runs the complete Phase 1 workflow for the EagleVision project on Kaggle.

It does all of the following in one place:

1. clone and install the repository
2. inspect the Kaggle dataset
3. normalize the dataset into the ScanNet-style layout expected by the repo
4. download a Depth Anything V2 checkpoint
5. generate Kaggle-local configs
6. run baseline evaluation
7. train the Phase 1 adaptation model
8. run adapted evaluation
9. compare baseline vs adapted metrics
10. export artifacts and a zip bundle

Confirmed dataset path for this notebook:

- `/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d`


## What Improvement We Are Testing

The proposed improvement is not a new learned renderer. The proposed improvement is a geometry-first training environment for improving a monocular depth estimator.

We compare:

- **baseline**: frozen Depth Anything V2
- **adapted**: Depth Anything V2 plus a lightweight residual adapter trained with round-trip geometric supervision

The key claim is that the adapted model should become more geometrically useful while preserving reasonable raw depth quality.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile

import yaml


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

REPO_URL = 'https://github.com/alooboii/EagleVision.git'
REPO_DIR = KAGGLE_WORKING / 'EagleVision'

# Confirmed dataset path for this Kaggle runtime
RAW_DATASET_DIR = Path('/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d')

print('KAGGLE_INPUT:', KAGGLE_INPUT)
print('KAGGLE_WORKING:', KAGGLE_WORKING)
print('RAW_DATASET_DIR:', RAW_DATASET_DIR)

if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset path not found: {RAW_DATASET_DIR}')


## Shell Helper

The helper below is deliberately non-fragile for Kaggle. It can capture stdout and stderr without hiding the actual failure reason when a command exits nonzero.


In [ ]:
def run(cmd, cwd=None, capture=False, check=True):
    print('$', ' '.join(cmd))
    print(f'[status] cwd={cwd if cwd else Path.cwd()}')
    print('[status] command started')

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    output_lines = []
    if process.stdout is not None:
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)

    return_code = process.wait()
    print(f'[status] command finished with exit code {return_code}')

    completed = subprocess.CompletedProcess(
        args=cmd,
        returncode=return_code,
        stdout=''.join(output_lines),
        stderr='',
    )

    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {' '.join(cmd)}"
        )
    return completed


def is_git_repo(path: Path) -> bool:
    return (path / '.git').exists()


## Clone Or Refresh The Repository

This cell is careful about Kaggle workspaces where a leftover directory may exist without actually being a git checkout.


In [ ]:
if REPO_DIR.exists() and not is_git_repo(REPO_DIR):
    print(f'Removing non-git directory at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if is_git_repo(REPO_DIR):
    run(['git', 'fetch', '--all'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])

run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR)
print('repo ready:', REPO_DIR)


In [ ]:
TRAIN_RUN_OUTPUT_DIR = REPO_DIR / 'outputs' / 'phase1_kaggle_complete'
CONFIG_DIR = REPO_DIR / 'outputs' / 'kaggle_configs'
TARGET_DATA_ROOT = REPO_DIR / 'data' / 'scannet'

for path in [TRAIN_RUN_OUTPUT_DIR, CONFIG_DIR, TARGET_DATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Make package imports robust in notebook kernels.
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ['PYTHONPATH'] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

print('TRAIN_RUN_OUTPUT_DIR:', TRAIN_RUN_OUTPUT_DIR)
print('CONFIG_DIR:', CONFIG_DIR)
print('TARGET_DATA_ROOT:', TARGET_DATA_ROOT)
print('SRC_DIR:', SRC_DIR)


## Inspect The Dataset

Before making any assumptions, inspect the first part of the directory tree. The screenshot you provided indicates scene folders with `color`, `depth`, `label`, and `pose`, which is exactly what we want.


In [ ]:
preview = [str(path) for path in sorted(RAW_DATASET_DIR.iterdir())[:80]]
preview


## Discover ScanNet-Style Scenes

We expect each scene to contain:

- `color/`
- `depth/`
- `pose/`

The extra `label/` directory is ignored.


In [ ]:
def discover_scene_dirs(root: Path):
    scenes = []
    for candidate in sorted(root.iterdir()):
        if not candidate.is_dir():
            continue
        if (candidate / 'color').exists() and (candidate / 'depth').exists() and (candidate / 'pose').exists():
            scenes.append(candidate)
    return scenes

scene_dirs = discover_scene_dirs(RAW_DATASET_DIR)
print('num scenes found:', len(scene_dirs))
print('first scenes:', [p.name for p in scene_dirs[:10]])

if not scene_dirs:
    raise RuntimeError('No valid scene directories found.')


## Normalize Into `data/scannet`

The repository expects data to live under `data/scannet`. We symlink scenes where possible and copy only if necessary.


In [ ]:
def materialize_scene(scene_dir: Path, target_root: Path):
    dst = target_root / scene_dir.name
    if dst.exists():
        return dst
    try:
        os.symlink(scene_dir, dst, target_is_directory=True)
    except OSError:
        shutil.copytree(scene_dir, dst)
    return dst

materialized = [materialize_scene(scene_dir, TARGET_DATA_ROOT) for scene_dir in scene_dirs]
scene_ids = sorted([p.name for p in TARGET_DATA_ROOT.iterdir() if p.is_dir()])

print('normalized scene count:', len(scene_ids))
print('example normalized scenes:', scene_ids[:10])


In [ ]:
first_scene = TARGET_DATA_ROOT / scene_ids[0]
print('scene:', first_scene.name)
print('color sample:', [p.name for p in sorted((first_scene / 'color').glob('*'))[:5]])
print('depth sample:', [p.name for p in sorted((first_scene / 'depth').glob('*'))[:5]])
print('pose sample:', [p.name for p in sorted((first_scene / 'pose').glob('*'))[:5]])


## Train / Validation Split

We create a simple scene-based split for the Kaggle run.


In [ ]:
all_scene_ids = list(scene_ids)
print('total normalized scenes available:', len(all_scene_ids))
print('scene sample:', all_scene_ids[:10])


## Download The Baseline Checkpoint

For Kaggle, we use the `vits` metric-depth model with the `hypersim` indoor profile.


In [ ]:
print('[status] downloading Depth Anything V2 checkpoints for vits (metric + relative)')
run(
    [
        'python', '-m', 'baseline.depth_anything_v2', 'download',
        '--mode', 'all',
        '--profile', 'hypersim',
        '--encoder', 'vits',
    ],
    cwd=REPO_DIR,
)


In [ ]:
checkpoint_dir = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints'
checkpoint_files = list(checkpoint_dir.glob('*.pth'))
print('checkpoint dir:', checkpoint_dir)
print('checkpoint files:', checkpoint_files)

if not checkpoint_files:
    raise RuntimeError('No checkpoint file found after download.')


## Build Kaggle-Local Train And Eval Configs

Edit only the next cell (`Single Control Cell`). Both baseline and adapted evaluation will use the same fair settings from that cell.


In [ ]:
# ===================== Single Control Cell (Edit Only Here) =====================
# Baseline vs adapted fairness rule in this notebook:
# 1) Same eval config file
# 2) Same scenes and pairing thresholds
# 3) Same image size and model settings
# 4) Same --max-batches cap
# 5) Only difference: baseline uses --baseline-only, adapted loads checkpoint

EXPERIMENT_TAG = 'phase1_kaggle_hour_fair'
SEED = 7

device_name = 'cuda' if shutil.which('nvidia-smi') else 'cpu'

# Compute/runtime budget
SCENE_LIMIT = 12
TRAIN_SCENE_FRACTION = 0.8
IMAGE_SIZE = [144, 192]

# Choose depth mode: 'metric' or 'relative'
# - metric: recommended for Phase 1 training with GT depth anchoring.
# - relative: works, but use geometry/RGB losses only (depth metric losses disabled below).
DEPTH_MODE = 'metric'
DEPTH_ENCODER = 'vits'
METRIC_PROFILE = 'hypersim'  # used only when DEPTH_MODE == 'metric'

# Pair sampling controls
PAIRING = {
    'min_translation_m': 0.03,
    'max_translation_m': 0.25,
    'min_rotation_deg': 1.0,
    'max_rotation_deg': 8.0,
    'max_index_gap': 8,
    'frame_stride': 4,
    'max_frames_per_scene': 100,
    'max_pairs_per_scene': 60,
}

# Resolve pretrained checkpoint according to selected mode.
if DEPTH_MODE == 'metric':
    ckpt_name = f'depth_anything_v2_metric_{METRIC_PROFILE}_{DEPTH_ENCODER}.pth'
else:
    ckpt_name = f'depth_anything_v2_{DEPTH_ENCODER}.pth'
DEFAULT_DEPTH_CKPT = REPO_DIR / 'baseline' / 'depth_anything_v2' / 'checkpoints' / ckpt_name
if not DEFAULT_DEPTH_CKPT.exists():
    raise FileNotFoundError(
        f'Missing pretrained DA-V2 checkpoint: {DEFAULT_DEPTH_CKPT}. Run the download cell first.'
    )

# Depth model controls (shared by train and eval)
DEPTH_MODEL_SETTINGS = {
    'mode': DEPTH_MODE,
    'encoder': DEPTH_ENCODER,
    'profile': METRIC_PROFILE,
    'checkpoint_path': str(DEFAULT_DEPTH_CKPT),
    'freeze_backbone': True,
    'adapter_hidden_channels': 32,
}

TRAIN_SETTINGS = {
    'batch_size': 1,
    'epochs': 1,
    'max_steps_per_epoch': 120,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'log_interval': 10,
    'vis_interval': 200,
    'checkpoint_interval': 50,
}

EVAL_SETTINGS = {
    'batch_size': 1,
    'max_batches': 80,
}

# In relative mode, disable metric-depth losses by default (scale ambiguity).
if DEPTH_MODE == 'relative':
    LOSS_WEIGHTS = {
        'target_rgb': 1.0,
        'cycle_rgb': 1.0,
        'cycle_depth': 0.0,
        'target_depth': 0.0,
    }
else:
    LOSS_WEIGHTS = {
        'target_rgb': 1.0,
        'cycle_rgb': 1.0,
        'cycle_depth': 0.5,
        'target_depth': 0.25,
    }

# ScanNet default intrinsics used by this notebook's resized samples
intrinsics = [
    [577.8706, 0.0, 159.5],
    [0.0, 577.8706, 119.5],
    [0.0, 0.0, 1.0],
]

scene_pool = list(all_scene_ids if 'all_scene_ids' in globals() else scene_ids)
if len(scene_pool) < 2:
    raise RuntimeError('Need at least 2 scenes for train/val split.')

if SCENE_LIMIT is not None:
    scene_ids = scene_pool[:SCENE_LIMIT]
else:
    scene_ids = scene_pool

split_index = max(1, int(TRAIN_SCENE_FRACTION * len(scene_ids)))
train_scenes = scene_ids[:split_index]
val_scenes = scene_ids[split_index:] if split_index < len(scene_ids) else scene_ids[:1]

TRAIN_RUN_OUTPUT_DIR = REPO_DIR / 'outputs' / EXPERIMENT_TAG
TRAIN_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_config = {
    'seed': SEED,
    'device': device_name,
    'output_dir': str(TRAIN_RUN_OUTPUT_DIR),
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': IMAGE_SIZE,
        'intrinsics': intrinsics,
        'pairing': dict(PAIRING),
        'splits': {
            'train': {'scenes': train_scenes},
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': dict(DEPTH_MODEL_SETTINGS),
    },
    'train': dict(TRAIN_SETTINGS),
    'eval': dict(EVAL_SETTINGS),
    'losses': {
        'weights': dict(LOSS_WEIGHTS),
    },
}

eval_config = {
    'seed': SEED,
    'device': device_name,
    'data': {
        'root': str(TARGET_DATA_ROOT),
        'image_size': IMAGE_SIZE,
        'intrinsics': intrinsics,
        'pairing': dict(PAIRING),
        'splits': {
            'val': {'scenes': val_scenes},
        },
    },
    'model': {
        'depth': dict(DEPTH_MODEL_SETTINGS),
    },
    'eval': dict(EVAL_SETTINGS),
    'losses': {
        'weights': dict(LOSS_WEIGHTS),
    },
}

train_config_path = CONFIG_DIR / f'train_{EXPERIMENT_TAG}.yaml'
eval_config_path = CONFIG_DIR / f'eval_{EXPERIMENT_TAG}.yaml'
with train_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(train_config, handle, sort_keys=False)
with eval_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(eval_config, handle, sort_keys=False)

FAIR_EVAL_ARGS = [
    '--config', str(eval_config_path),
    '--max-batches', str(EVAL_SETTINGS['max_batches']),
]

print('device:', device_name)
print('train config:', train_config_path)
print('eval config:', eval_config_path)
print('train scenes:', len(train_scenes), 'val scenes:', len(val_scenes))
print('fair eval args:', FAIR_EVAL_ARGS)
print('depth mode:', DEPTH_MODE)
print('shared depth settings:', DEPTH_MODEL_SETTINGS)
print('pretrained depth checkpoint:', DEFAULT_DEPTH_CKPT)
print('loss weights:', LOSS_WEIGHTS)


## Metric Parser

The evaluation CLI prints metrics to stdout. This helper turns that text into a dictionary for comparison and export.


In [ ]:
def parse_metric_output(stdout_text: str):
    metrics = {}
    for line in stdout_text.splitlines():
        if ': ' not in line:
            continue
        key, value = line.split(': ', 1)
        try:
            metrics[key.strip()] = float(value.strip())
        except ValueError:
            continue
    return metrics


## Baseline Evaluation

This measures the frozen baseline before any adaptation.


In [ ]:
print('[status] starting baseline evaluation (fair settings)')
baseline_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', *FAIR_EVAL_ARGS, '--baseline-only'],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('baseline return code:', baseline_eval.returncode)
if baseline_eval.returncode != 0:
    raise RuntimeError('Baseline evaluation failed. Read streamed output above.')


In [ ]:
baseline_metrics = parse_metric_output(baseline_eval.stdout)
baseline_metrics_path = TRAIN_RUN_OUTPUT_DIR / 'baseline_metrics.yaml'

with baseline_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(baseline_metrics, handle, sort_keys=False)

baseline_metrics


## Train The Phase 1 Adaptation Model

This is the proposed improvement step: geometry-first round-trip supervision over a mostly frozen depth backbone with a lightweight adaptation head.


In [ ]:
print('[status] starting Phase 1 training run')
train_run = run(
    ['python', '-m', 'eaglevision.cli.train', '--config', str(train_config_path)],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('train return code:', train_run.returncode)
if train_run.returncode != 0:
    raise RuntimeError('Training failed. Read streamed output above.')


In [ ]:
checkpoint_dir = TRAIN_RUN_OUTPUT_DIR / 'checkpoints'
checkpoints = sorted(checkpoint_dir.glob('*.pt'))
print('checkpoints:', checkpoints)

if not checkpoints:
    raise RuntimeError(f'No checkpoints found in {checkpoint_dir}')

latest_checkpoint = checkpoints[-1]
latest_checkpoint


## Adapted Evaluation

This measures the trained adapted model using the latest checkpoint from the training run.


In [ ]:
print('[status] starting adapted evaluation (same fair settings)')
adapted_eval = run(
    ['python', '-m', 'eaglevision.cli.eval', *FAIR_EVAL_ARGS, '--checkpoint', str(latest_checkpoint)],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('adapted return code:', adapted_eval.returncode)
if adapted_eval.returncode != 0:
    raise RuntimeError('Adapted evaluation failed. Read streamed output above.')


In [ ]:
adapted_metrics = parse_metric_output(adapted_eval.stdout)
adapted_metrics_path = TRAIN_RUN_OUTPUT_DIR / 'adapted_metrics.yaml'

with adapted_metrics_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(adapted_metrics, handle, sort_keys=False)

adapted_metrics


## Compare Baseline And Adapted Metrics

This is the central proof step for the proposed improvement.

Interpretation:

- lower is better for `target_rgb_l1`, `cycle_rgb_l1`, `reprojection_depth`, `depth_l1`, `rmse`, `abs_rel`
- higher is better for `psnr` and `ssim`


In [ ]:
metric_names = sorted(set(baseline_metrics) | set(adapted_metrics))
comparison_rows = []

for metric_name in metric_names:
    baseline_value = baseline_metrics.get(metric_name)
    adapted_value = adapted_metrics.get(metric_name)
    delta = None
    if baseline_value is not None and adapted_value is not None:
        delta = adapted_value - baseline_value
    comparison_rows.append(
        {
            'metric': metric_name,
            'baseline': baseline_value,
            'adapted': adapted_value,
            'delta': delta,
        }
    )

comparison_rows


In [ ]:
comparison_json_path = TRAIN_RUN_OUTPUT_DIR / 'comparison_metrics.json'
comparison_csv_path = TRAIN_RUN_OUTPUT_DIR / 'comparison_metrics.csv'

with comparison_json_path.open('w', encoding='utf-8') as handle:
    json.dump(comparison_rows, handle, indent=2)

with comparison_csv_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['metric', 'baseline', 'adapted', 'delta'])
    writer.writeheader()
    writer.writerows(comparison_rows)

print(comparison_json_path)
print(comparison_csv_path)


## Result Analysis And Consolidation Plots

These cells summarize whether adapted metrics improved in the expected direction and visualize effect size.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(comparison_rows)
df = df.sort_values('metric').reset_index(drop=True)

lower_better = {
    'target_rgb_l1', 'cycle_rgb_l1', 'reprojection_depth',
    'depth_l1', 'rmse', 'abs_rel',
    'loss_total', 'loss_target_rgb', 'loss_cycle_rgb', 'loss_cycle_depth', 'loss_target_depth',
}
higher_better = {'psnr', 'ssim'}

def direction(metric: str) -> str:
    if metric in lower_better:
        return 'lower_better'
    if metric in higher_better:
        return 'higher_better'
    return 'unknown'

df['direction'] = df['metric'].map(direction)
df['improved'] = np.where(
    df['direction'].eq('lower_better'), df['delta'] < 0,
    np.where(df['direction'].eq('higher_better'), df['delta'] > 0, np.nan)
)
df['rel_change_pct'] = 100.0 * (df['adapted'] - df['baseline']) / (df['baseline'].replace(0, np.nan))

tracked = df[df['direction'] != 'unknown'].copy()
num_improved = int(tracked['improved'].fillna(False).sum())
num_total = int(len(tracked))
print(f'Improved metrics (direction-aware): {num_improved}/{num_total}')
display(df[['metric','baseline','adapted','delta','rel_change_pct','direction','improved']])


In [ ]:
plot_df = df[df['delta'].notna()].copy()
plot_df = plot_df.sort_values('delta')

colors = []
for _, row in plot_df.iterrows():
    if row['direction'] == 'lower_better':
        colors.append('#2ca02c' if row['delta'] < 0 else '#d62728')
    elif row['direction'] == 'higher_better':
        colors.append('#2ca02c' if row['delta'] > 0 else '#d62728')
    else:
        colors.append('#7f7f7f')

plt.figure(figsize=(10, max(6, 0.4 * len(plot_df))))
plt.barh(plot_df['metric'], plot_df['delta'], color=colors)
plt.axvline(0.0, color='black', linewidth=1)
plt.title('Adapted - Baseline Delta By Metric')
plt.xlabel('Delta')
plt.tight_layout()
plt.show()


In [ ]:
focus_metrics = [
    'target_rgb_l1', 'cycle_rgb_l1', 'reprojection_depth',
    'depth_l1', 'rmse', 'abs_rel', 'psnr', 'ssim',
]
focus = df[df['metric'].isin(focus_metrics)].copy()
focus = focus.set_index('metric').loc[[m for m in focus_metrics if m in set(focus['metric'])]].reset_index()

x = np.arange(len(focus))
w = 0.38
plt.figure(figsize=(12, 5))
plt.bar(x - w/2, focus['baseline'], width=w, label='Baseline', color='#1f77b4')
plt.bar(x + w/2, focus['adapted'], width=w, label='Adapted', color='#ff7f0e')
plt.xticks(x, focus['metric'], rotation=35, ha='right')
plt.title('Baseline vs Adapted (Key Metrics)')
plt.legend()
plt.tight_layout()
plt.show()

print('Direction reference: lower is better for L1/RMSE/AbsRel/reprojection; higher is better for PSNR/SSIM.')


In [ ]:
# Optional: aggregate multiple run comparisons if present under outputs/**/comparison_metrics.json
all_cmp_files = sorted((REPO_DIR / 'outputs').rglob('comparison_metrics.json'))
print('found comparison files:', len(all_cmp_files))
for pth in all_cmp_files[:10]:
    print('-', pth)

rows = []
for cmp_path in all_cmp_files:
    run_name = cmp_path.parent.name
    with cmp_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    for item in data:
        rows.append({'run': run_name, **item})

if rows:
    runs_df = pd.DataFrame(rows)
    tracked = runs_df[runs_df['metric'].isin(focus_metrics)].copy()
    if not tracked.empty:
        summary = tracked.groupby('metric')['delta'].agg(['count','mean','std']).reset_index()
        display(summary.sort_values('metric'))

        plt.figure(figsize=(11, 4))
        ordered = summary.sort_values('mean')
        plt.bar(ordered['metric'], ordered['mean'], color='#1f77b4')
        plt.axhline(0.0, color='black', linewidth=1)
        plt.xticks(rotation=35, ha='right')
        plt.title('Mean Delta Across Runs (Adapted - Baseline)')
        plt.tight_layout()
        plt.show()
    else:
        print('No focus metrics found in aggregated run files.')
else:
    print('No comparison files found yet. Re-run this cell after multiple experiments.')


## Baseline vs Adapted Depth Visualization (Input + Novel View)

This section shows pure depth estimation visuals only (no difference heatmaps):
- source input RGB: baseline depth vs adapted depth
- forward-warped novel RGB: baseline depth vs adapted depth

Note: forward warp is sparse geometry, so black regions in novel RGB are expected holes (no inpainting in Phase 1).
In relative mode, absolute numeric depth is arbitrary; compare spatial structure rather than absolute scale.



In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from eaglevision.data.pair_sampler import PairSamplingConfig
from eaglevision.data.scannet_dataset import ScanNetPairDataset
from eaglevision.engine.checkpointing import load_checkpoint
from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.nvs.geometric_warp import GeometricWarper
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS

viz_device = torch.device(device_name if torch.cuda.is_available() else 'cpu')

if 'latest_checkpoint' not in globals() or latest_checkpoint is None:
    checkpoint_candidates = sorted((TRAIN_RUN_OUTPUT_DIR / 'checkpoints').glob('*.pt'))
    if not checkpoint_candidates:
        raise RuntimeError('No training checkpoint available for adapted-depth visualization.')
    latest_checkpoint = checkpoint_candidates[-1]

if DEPTH_MODEL_SETTINGS.get('checkpoint_path') is None:
    raise RuntimeError('DEPTH_MODEL_SETTINGS.checkpoint_path is None. Baseline would not be pretrained.')

pair_cfg = PairSamplingConfig(**PAIRING)
base_intrinsics = np.array(intrinsics, dtype=np.float32)

viz_scenes = list(val_scenes)
if not viz_scenes:
    viz_scenes = list(train_scenes[:1])

viz_dataset = ScanNetPairDataset(
    root=TARGET_DATA_ROOT,
    scenes=viz_scenes,
    image_size=tuple(IMAGE_SIZE),
    intrinsics=base_intrinsics,
    pair_config=pair_cfg,
)
if len(viz_dataset) == 0:
    viz_dataset = ScanNetPairDataset(
        root=TARGET_DATA_ROOT,
        scenes=list(train_scenes),
        image_size=tuple(IMAGE_SIZE),
        intrinsics=base_intrinsics,
        pair_config=pair_cfg,
    )
if len(viz_dataset) == 0:
    raise RuntimeError('No valid pairs available for visualization. Relax pairing controls in the single control cell.')

sample = viz_dataset[0]

source_rgb = sample['source_rgb'].unsqueeze(0).to(viz_device)
source_depth = sample['source_depth'].unsqueeze(0).to(viz_device)
source_k = sample['source_intrinsics'].unsqueeze(0).to(viz_device)
target_k = sample['target_intrinsics'].unsqueeze(0).to(viz_device)
source_pose = sample['source_pose'].unsqueeze(0).to(viz_device)
target_pose = sample['target_pose'].unsqueeze(0).to(viz_device)

t_s2t = target_pose @ torch.inverse(source_pose)
warper = GeometricWarper().to(viz_device).eval()

base_depth_model = DepthAnythingWithAdapter(**DEPTH_MODEL_SETTINGS).to(viz_device).eval()
for parameter in base_depth_model.adapter.parameters():
    parameter.data.zero_()
    parameter.requires_grad = False

adapted_depth_model = DepthAnythingWithAdapter(**DEPTH_MODEL_SETTINGS).to(viz_device)
adapted_roundtrip = RoundTripDepthNVS(adapted_depth_model).to(viz_device).eval()
load_checkpoint(latest_checkpoint, adapted_roundtrip)

with torch.no_grad():
    forward = warper(source_rgb, source_depth, source_k, target_k, t_s2t)
    novel_rgb = forward['warped_rgb']

    baseline_source_pack = base_depth_model(source_rgb)
    baseline_novel_pack = base_depth_model(novel_rgb)

    adapted_source_pack = adapted_roundtrip.depth_model(source_rgb)
    adapted_novel_pack = adapted_roundtrip.depth_model(novel_rgb)

    source_depth_baseline = baseline_source_pack['adapted_depth'][0].detach().cpu().numpy()
    source_depth_adapted = adapted_source_pack['adapted_depth'][0].detach().cpu().numpy()

    novel_depth_baseline = baseline_novel_pack['adapted_depth'][0].detach().cpu().numpy()
    novel_depth_adapted = adapted_novel_pack['adapted_depth'][0].detach().cpu().numpy()

source_rgb_np = source_rgb[0].detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)
novel_rgb_np = novel_rgb[0].detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)

joint = np.concatenate(
    [
        source_depth_baseline.reshape(-1),
        source_depth_adapted.reshape(-1),
        novel_depth_baseline.reshape(-1),
        novel_depth_adapted.reshape(-1),
    ]
)
finite = np.isfinite(joint)
vmin, vmax = (np.percentile(joint[finite], 2), np.percentile(joint[finite], 98)) if finite.any() else (0.0, 1.0)

baseline_zero_adapter_source_mae = float(
    (baseline_source_pack['adapted_depth'] - baseline_source_pack['base_depth']).abs().mean().item()
)
baseline_zero_adapter_novel_mae = float(
    (baseline_novel_pack['adapted_depth'] - baseline_novel_pack['base_depth']).abs().mean().item()
)
adapted_residual_source_mae = float(
    (adapted_source_pack['adapted_depth'] - adapted_source_pack['base_depth']).abs().mean().item()
)
adapted_residual_novel_mae = float(
    (adapted_novel_pack['adapted_depth'] - adapted_novel_pack['base_depth']).abs().mean().item()
)

print('checkpoint (adapted model):', latest_checkpoint)
print('backbone preload checkpoint:', DEPTH_MODEL_SETTINGS['checkpoint_path'])
print('mode:', DEPTH_MODEL_SETTINGS['mode'])
print('scene:', sample['scene_id'])
print('source frame:', sample['source_frame_key'], 'target frame:', sample['target_frame_key'])
print('baseline zero-adapter MAE source:', baseline_zero_adapter_source_mae)
print('baseline zero-adapter MAE novel :', baseline_zero_adapter_novel_mae)
print('adapted residual MAE source     :', adapted_residual_source_mae)
print('adapted residual MAE novel      :', adapted_residual_novel_mae)
print('source depth mean baseline/adapted:', float(np.nanmean(source_depth_baseline)), float(np.nanmean(source_depth_adapted)))
print('novel depth mean baseline/adapted :', float(np.nanmean(novel_depth_baseline)), float(np.nanmean(novel_depth_adapted)))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)

axes[0, 0].imshow(source_rgb_np)
axes[0, 0].set_title('Source Input RGB')
axes[0, 0].axis('off')

im_a = axes[0, 1].imshow(source_depth_baseline, cmap='magma', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('Source Depth: Baseline')
axes[0, 1].axis('off')

axes[0, 2].imshow(source_depth_adapted, cmap='magma', vmin=vmin, vmax=vmax)
axes[0, 2].set_title('Source Depth: Adapted')
axes[0, 2].axis('off')

axes[1, 0].imshow(novel_rgb_np)
axes[1, 0].set_title('Novel View RGB (Forward Warp)')
axes[1, 0].axis('off')

axes[1, 1].imshow(novel_depth_baseline, cmap='magma', vmin=vmin, vmax=vmax)
axes[1, 1].set_title('Novel View Depth: Baseline')
axes[1, 1].axis('off')

axes[1, 2].imshow(novel_depth_adapted, cmap='magma', vmin=vmin, vmax=vmax)
axes[1, 2].set_title('Novel View Depth: Adapted')
axes[1, 2].axis('off')

fig.colorbar(
    im_a,
    ax=[axes[0, 1], axes[0, 2], axes[1, 1], axes[1, 2]],
    fraction=0.025,
    pad=0.02,
    label='Depth',
)

depth_compare_panel = TRAIN_RUN_OUTPUT_DIR / 'baseline_vs_adapted_input_novel_depth_panel.png'
fig.savefig(depth_compare_panel, dpi=150)
plt.show()
print('saved:', depth_compare_panel)


## Optional Inference Artifact

This produces one depth-preview artifact from the normalized dataset.


In [ ]:
first_scene = TARGET_DATA_ROOT / scene_ids[0]
sample_image = next((first_scene / 'color').glob('*.jpg'))
infer_output = TRAIN_RUN_OUTPUT_DIR / 'sample_infer_depth.png'

infer_run = run(
    [
        'python', '-m', 'eaglevision.cli.infer_depth',
        '--input', str(sample_image),
        '--output', str(infer_output),
        '--mode', 'metric',
        '--encoder', 'vits',
        '--profile', 'hypersim',
    ],
    cwd=REPO_DIR,
    capture=True,
    check=False,
)

print('infer return code:', infer_run.returncode)
infer_output


## Produced Artifacts

List the main artifacts generated by this run.


In [ ]:
artifact_paths = sorted(str(path.relative_to(REPO_DIR)) for path in TRAIN_RUN_OUTPUT_DIR.rglob('*'))
artifact_paths[:200]


## Package Outputs For Export

This creates a zip file containing the run artifacts, metric summaries, and checkpoint outputs for later reuse.


In [ ]:
archive_base = REPO_DIR / 'outputs' / 'phase1_kaggle_complete_export'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=TRAIN_RUN_OUTPUT_DIR)
archive_path


## Final Summary

This notebook demonstrates the full proposed Phase 1 workflow:

- baseline depth evaluation
- geometry-first adaptation training
- adapted evaluation
- direct baseline vs adapted comparison

If the adapted model improves the geometry-facing metrics while maintaining acceptable depth quality, that supports the central project hypothesis.
